In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\data\Telco-Customer-Churn.csv')

In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [3]:
df = df.drop("customerID", axis=1)

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df_encoded = pd.get_dummies(df, drop_first=True)

In [4]:
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]


In [5]:
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42))
])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_prob = np.zeros(X_train.shape[0])

for train_idx, val_idx in cv.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    pipeline_lr.fit(X_tr, y_tr)
    oof_prob[val_idx] = pipeline_lr.predict_proba(X_val)[:, 1]
    

In [6]:
oof_prob

array([0.38202166, 0.2934312 , 0.03903892, ..., 0.55979095, 0.03344909,
       0.15583269], shape=(5634,))

In [18]:
from sklearn.metrics import f1_score, roc_auc_score

thresholds = np.round(np.arange(0.05, 0.91, 0.05), 2)

results = []

for threshold in thresholds:
    oof_pred = (oof_prob >= threshold).astype(int)
    f1 = f1_score(y_train, oof_pred)
    roc_auc = roc_auc_score(y_train, oof_prob)
    results.append({
        "threhold": threshold,
        "f1": f1,
        "roc_auc": roc_auc
    })

results_df = pd.DataFrame(results)
results_df.sort_values(by="f1", ascending=False, inplace=True)
results_df

,threhold,f1,roc_auc
5,0.30,0.634578,0.845676
6,0.35,0.633481,0.845676
4,0.25,0.627119,0.845676
7,0.40,0.627092,0.845676
8,0.45,0.612189,0.845676
3,0.20,0.611416,0.845676
9,0.50,0.594240,0.845676
2,0.15,0.592753,0.845676
10,0.55,0.570086,0.845676
1,0.10,0.561389,0.845676


In [14]:
best_row = results_df.loc[
    results_df["f1"].idxmax()
]

best_threshold = best_row["threhold"]

print(f"Best Threshold: {best_threshold}, Best F1 Score: {best_row['f1']:.4f}")

Best Threshold: 0.3, Best F1 Score: 0.6346


In [15]:
final_model = pipeline_lr.fit(X_train, y_train)

test_prob = final_model.predict_proba(X_test)[:, 1]

In [16]:
y_pred = (test_prob >= best_threshold).astype(int)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, test_prob)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}")
print(f"Confusion Matrix:\n{cm}")

Accuracy: 0.7495
Precision: 0.5193
Recall: 0.7540
F1 Score: 0.6150
Confusion Matrix:
[[774 261]
 [ 92 282]]
